# Data pipeline walkthrough

This notebook walks through `data_cleaning/`, the pipeline that turns the ideal bond universe (iShares AGG holdings
in `data/AGG_holdings_clean.csv`) into daily, cleaned, feature-engineered bond data from WRDS. Each section runs
one step of the pipeline on real data (cached pulls in `data/raw/`, so it does not hit WRDS unless a cache is missing),
and the last section looks at the finished tables in `data/processed/`.

To (re)build everything from the command line:

```sh
uv run python -m data_cleaning.run_pipeline      # ~25 min first time (WRDS pulls), ~15 min from caches
uv run python -m data_cleaning.validate          # sanity checks
```

| step | module | source | output |
|---|---|---|---|
| 1 | `universe.py` | AGG holdings CSV | one row per bond in the modelled sectors |
| 2 | `treasuries.py` | CRSP `tfz_dly` / `tfz_iss` | daily Treasury bid/ask/mid, accrued, yield |
| 3 | `curve.py` | step 2 | daily NSS zero curve, zero yields at 8 key rates |
| 4 | `trace_clean.py`, `corporates.py`, `mbs.py` | TRACE enhanced, FISD, Dickerson | daily corporate and MBS prices, volume, bid-ask |
| 5 | `features.py` | steps 2-4 | solved yield to maturity, spread over the zero curve |
| 6 | `merge.py` | all | `bond_universe`, `prices_daily`, `yield_curve_daily`, ... as parquet |

In [1]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))        # repo root, so `data_cleaning` imports
from data_cleaning import config as C, trace_clean
from data_cleaning.corporates import clean_terms, pull_dickerson, pull_fisd_terms
from data_cleaning.curve import KEY_COLS, fit_curves
from data_cleaning.features import solve_yield_and_spread
from data_cleaning.treasuries import clean_treasuries, pull_treasuries
from data_cleaning.universe import build_universe

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
pd.set_option("display.width", 200, "display.max_columns", 40, "display.float_format", "{:,.4f}".format)
print("window:", C.START_DATE, "to", C.END_DATE)

window: 2024-01-01 to 2026-03-31


## Step 1: the bond universe

The AGG file is the ideal universe. We keep the sectors the model covers (`config.SECTOR_MAP`): Treasuries,
industrials, financials, utilities and MBS pass-throughs. The other sectors (CMBS, ABS, sovereign, supranational,
agency, local authority, ~5% of the fund) have no daily WRDS source we process.

Note the file's `Weight (%)` column is rounded to two decimals, which zeroes most corporates, so weights are
recomputed from market value in the merge step.

In [2]:
universe = build_universe()
summary = universe.groupby("sector").agg(bonds=("cusip", "size"), market_value_bn=("agg_market_value", lambda s: s.sum() / 1e9),
                                         file_weight_pct=("agg_weight_pct", "sum"))
summary["mv_share"] = summary["market_value_bn"] / summary["market_value_bn"].sum()
summary

data_cleaning.universe: AGG holdings as of 2026-10-12: fund weight 81.9%, modelled sectors 77.5% in 12069 bonds


,bonds,market_value_bn,file_weight_pct,mv_share
sector,,,,
financial,2347,10.9843,4.5200,0.0849
industrial,5032,19.5271,4.8700,0.1509
mbs_passthrough,2934,31.2298,21.1600,0.2413
treasury,297,64.2729,46.7900,0.4966
utility,1459,3.4246,0.1200,0.0265


In [3]:
universe[["cusip", "name", "sector", "agg_market_value", "agg_maturity_date", "agg_coupon_pct", "agg_duration", "agg_ytm_pct", "is_tba"]].head(8)

,cusip,name,sector,agg_market_value,agg_maturity_date,agg_coupon_pct,agg_duration,agg_ytm_pct,is_tba
0,91282CGQ8,TREASURY NOTE,treasury,"545,793,898.2500",2030-02-28,4.0000,3.1000,4.9500,False
1,91282CMM0,TREASURY NOTE,treasury,"545,515,407.5800",2035-02-15,4.6300,6.7500,5.2000,False
2,91282CQQ7,TREASURY NOTE (OLD),treasury,"544,523,031.4000",2036-05-15,4.3800,7.5100,5.2400,False
3,91282CLW9,TREASURY NOTE,treasury,"543,314,740.0800",2034-11-15,4.2500,6.5900,5.1900,False
4,91282CKQ3,TREASURY NOTE,treasury,"540,705,529.9700",2034-05-15,4.3800,6.2200,5.1700,False
5,91282CNC1,TREASURY NOTE,treasury,"533,597,132.9600",2035-05-15,4.2500,6.9100,5.2100,False
6,91282CNM9,TREASURY NOTE,treasury,"528,596,128.9900",2028-07-15,3.8800,1.6700,4.7700,False
7,91282CLF6,TREASURY NOTE,treasury,"524,926,836.2200",2034-08-15,3.8800,6.5600,5.1900,False


## Step 2: CRSP Treasuries

CRSP quotes every Treasury every trading day (bid, ask, accrued interest, yield, duration). Cleaning:
TIPS and callables dropped, `clean_price` = (bid + ask) / 2, `dirty_price` = clean + CRSP accrued, and the 8-character
CRSP CUSIP gets its check digit so it joins to the AGG 9-character CUSIP. All nominal Treasuries are kept here
because the curve fit (step 3) uses the whole market, not just AGG's holdings.

In [4]:
tsy_all = clean_treasuries(pull_treasuries())
tsy_all[["cusip", "date", "security_type", "coupon_rate", "maturity_date", "bid_price", "ask_price", "clean_price",
         "accrued_interest", "dirty_price", "bid_ask_spread", "yield", "duration"]].head()

data_cleaning.wrds_pull: crsp_treasuries: 223858 rows from cache


,cusip,date,security_type,coupon_rate,maturity_date,bid_price,ask_price,clean_price,accrued_interest,dirty_price,bid_ask_spread,yield,duration
0,912796CX5,2024-01-02,4,0.0000,2024-04-18,98.4574,98.4604,98.4589,0.0000,98.4589,0.0030,0.0530,0.2932
1,912796CX5,2024-01-03,4,0.0000,2024-04-18,98.4704,98.4733,98.4718,0.0000,98.4718,0.0029,0.0530,0.2904
2,912796CX5,2024-01-04,4,0.0000,2024-04-18,98.4892,98.4921,98.4906,0.0000,98.4906,0.0029,0.0529,0.2877
3,912796CX5,2024-01-05,4,0.0000,2024-04-18,98.5036,98.5064,98.5050,0.0000,98.5050,0.0029,0.0529,0.2849
4,912796CX5,2024-01-08,4,0.0000,2024-04-18,98.5439,98.5467,98.5453,0.0000,98.5453,0.0028,0.0530,0.2767


In [5]:
agg_tsy = set(universe.loc[universe["asset_type"] == "treasury", "cusip"])
found = agg_tsy & set(tsy_all["cusip"])
missing = universe[universe["cusip"].isin(agg_tsy - found)]
print(f"{len(tsy_all):,} quote rows on {tsy_all['date'].nunique()} trading days, {tsy_all['date'].min().date()} to {tsy_all['date'].max().date()}")
print(f"AGG Treasuries found in CRSP: {len(found)} of {len(agg_tsy)}; missing ones are all issued after CRSP's last date:")
missing[["name", "cusip", "agg_maturity_date", "agg_accrual_date"]].sort_values("agg_accrual_date").head()

197,502 quote rows on 499 trading days, 2024-01-02 to 2025-12-31
AGG Treasuries found in CRSP: 253 of 297; missing ones are all issued after CRSP's last date:


,name,cusip,agg_maturity_date,agg_accrual_date
171,TREASURY NOTE,91282CPT2,2029-01-15,2026-01-15
203,TREASURY NOTE,91282CPY1,2033-01-31,2026-01-31
45,TREASURY NOTE,91282CPW5,2031-01-31,2026-01-31
9,TREASURY NOTE (2OLD),91282CPZ8,2036-02-15,2026-02-15
266,TREASURY BOND (2OLD),912810UT3,2046-02-15,2026-02-15


## Step 3: the Treasury zero curve

For each trading day a Nelson-Siegel-Svensson zero curve is fitted to the dirty prices of all nominal
Treasuries (duration-weighted least squares, with one robust pass that zero-weights outlier bonds). The zero yields
at the 8 key-rate tenors (6M ... 30Y, continuously compounded, decimal) are the model's curve factors; their daily
changes feed the key-rate covariance.

In [6]:
curve = fit_curves(tsy_all)
curve[["date"] + KEY_COLS + ["fit_rmse_price", "n_bonds_used", "n_bonds_quoted"]].tail()

data_cleaning.curve: fitted 499 daily curves, median price rmse 0.035


,date,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y,fit_rmse_price,n_bonds_used,n_bonds_quoted
494,2025-12-24,0.0352,0.0350,0.0350,0.0354,0.0370,0.0391,0.0423,0.0497,0.0484,317,401
495,2025-12-26,0.0352,0.0350,0.0345,0.0349,0.0369,0.0392,0.0423,0.0512,0.0571,312,400
496,2025-12-29,0.0352,0.0348,0.0345,0.0348,0.0365,0.0387,0.0421,0.0501,0.0362,310,400
497,2025-12-30,0.0353,0.0347,0.0343,0.0347,0.0367,0.0390,0.0421,0.0514,0.0384,293,400
498,2025-12-31,0.0353,0.0348,0.0345,0.0352,0.0372,0.0395,0.0425,0.0526,0.0244,298,400


In [7]:
# month-end snapshot of the curve, in percent
me = curve.groupby(curve["date"].dt.to_period("M")).tail(1).set_index("date")[KEY_COLS] * 100
me.round(2).iloc[::3]

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y
date,,,,,,,,
2024-01-31,5.0100,4.6400,4.1600,3.9200,3.7900,3.8400,3.9800,4.3300
2024-04-30,5.2700,5.1600,4.9700,4.8300,4.6700,4.6200,4.6400,4.7400
2024-07-31,5.0000,4.6600,4.2300,4.0100,3.8700,3.9100,4.0400,4.4300
2024-10-31,4.4000,4.2500,4.1200,4.0900,4.1300,4.2000,4.3000,4.6800
2025-01-31,4.2200,4.1800,4.1600,4.2000,4.3100,4.4200,4.5600,5.0000
2025-04-30,4.1000,3.8400,3.5700,3.5500,3.7100,3.9200,4.2200,4.9800
2025-07-31,4.2400,4.1200,3.9000,3.8400,3.9300,4.1200,4.4000,5.1400
2025-10-31,3.7600,3.6800,3.5600,3.5600,3.7000,3.8800,4.1400,5.0500


In [8]:
# daily changes of the key rates (bps): the input to the covariance estimate in Deliverable 3, Section 2.1
d_key = (curve.set_index("date")[KEY_COLS].diff() * 1e4).dropna()
d_key.describe().round(2)

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y
count,498.0000,498.0000,498.0000,498.0000,498.0000,498.0000,498.0000,498.0000
mean,-0.3100,-0.2500,-0.1700,-0.1000,-0.0300,0.0100,0.0500,0.2100
std,2.7100,3.9500,5.3800,5.6800,5.5700,5.4700,5.5400,8.5300
min,-18.0800,-28.3900,-29.6800,-24.5500,-20.0100,-20.3000,-22.1600,-35.6400
25%,-1.6100,-2.1900,-3.1000,-3.3700,-3.5700,-3.6200,-3.4400,-4.4000
50%,-0.3100,-0.1600,-0.2300,-0.3200,-0.0700,-0.0800,-0.0500,-0.1500
75%,1.0300,1.7900,3.0200,3.4200,3.2200,3.4600,3.5400,4.7600
max,12.2900,17.5600,22.5000,24.4500,22.8200,19.7100,19.0500,32.9600


## Step 4a: corporates from enhanced TRACE

TRACE is trade-level, so each day's price has to be built from the raw trade reports. The raw pull is cached per
chunk of 250 CUSIPs in `data/raw/trace_corporate/`; this section cleans one chunk to show the steps:

1. cancellations/corrections (`trc_st` C/X, which carry the cancelled report's `msg_seq_nb`) and reversals (Y) are dropped
   together with the reports they refer to; corrected replacements (R) are kept
2. inter-dealer trades are reported by both dealers; the buy-side copy is dropped
3. when-issued, locked-in, special-condition and settlement > T+2 trades are dropped
4. Edwards-Harris-Piwowar price filters: within 10% of the daily and 9-day medians; no isolated 10% reversals

Then one row per bond-day: face-weighted price, volume, trade count, and the dealer bid/ask implied by customer trades.

In [9]:
chunk = sorted((C.RAW_DIR / "trace_corporate").glob("chunk_010_*.parquet"))[0]
raw = pd.read_parquet(chunk)
print(f"{len(raw):,} raw trade reports for {raw['cusip_id'].nunique()} bonds")
print(raw["trc_st"].value_counts().to_dict())
raw[["cusip_id", "trd_exctn_dt", "trd_exctn_tm", "msg_seq_nb", "orig_msg_seq_nb", "trc_st", "rptd_pr", "entrd_vol_qt",
     "yld_pt", "rpt_side_cd", "cntra_mp_id", "stlmnt_dt"]].head(8)

1,452,543 raw trade reports for 221 bonds
{'T': 1441628, 'C': 3822, 'R': 3816, 'X': 3210, 'Y': 67}


,cusip_id,trd_exctn_dt,trd_exctn_tm,msg_seq_nb,orig_msg_seq_nb,trc_st,rptd_pr,entrd_vol_qt,yld_pt,rpt_side_cd,cntra_mp_id,stlmnt_dt
0,21685WCJ4,2024-01-02,11:02:12,0048081,<NA>,T,104.7530,"50,000.0000",4.8421,B,C,2024-01-03
1,21685WCJ4,2024-01-02,11:19:39,0056117,<NA>,T,105.1270,"125,000.0000",4.8110,B,D,2024-01-04
2,21685WCJ4,2024-01-02,11:19:39,0056118,<NA>,T,105.1270,"125,000.0000",4.8110,S,C,2024-01-04
3,21685WCJ4,2024-01-02,11:19:39,0056122,<NA>,T,105.1270,"125,000.0000",4.8110,S,D,2024-01-04
4,21685WCJ4,2024-01-02,13:16:17,0106208,<NA>,T,105.0310,"250,000.0000",4.8190,S,C,2024-01-04
5,21685WCJ4,2024-01-02,17:24:41,0196578,<NA>,T,104.8970,"60,000.0000",4.8301,S,A,2024-01-04
6,21685WCJ4,2024-01-03,16:00:12,0196957,<NA>,T,105.0540,"50,000.0000",4.8170,B,D,2024-01-05
7,21685WCJ4,2024-01-03,16:00:12,0196970,<NA>,T,104.9940,"50,000.0000",4.8220,B,C,2024-01-05


In [10]:
trades, step_log = trace_clean.clean_trades(raw, product="corporate")
step_log["dropped"] = -step_log["trades_left"].diff()
step_log

,step,trades_left,dropped
0,0 valid raw trades,1452543,NaN
1,1 cancels/corrections/reversals,1438348,"14,195.0000"
2,2 inter-dealer duplicates,1091388,"346,960.0000"
3,3 non-standard trades,1037989,"53,399.0000"
4,4a median filter,1037988,1.0000
5,4b reversal filter,1037988,-0.0000


In [11]:
corp_daily = trace_clean.daily_prices(trades)
print(f"{len(corp_daily):,} bond-days for {corp_daily['cusip'].nunique()} bonds")
corp_daily.head()

85,124 bond-days for 221 bonds


,cusip,date,clean_price,yield,price_low,price_high,par_volume,trade_count,dollar_volume,price_bid,price_ask,bid_ask_spread
0,21685WCJ4,2024-01-02,105.0344,0.0482,104.7530,105.1270,"610,000.0000",5,"640,709.7000",104.7530,105.0630,0.3100
1,21685WCJ4,2024-01-03,105.0240,0.0482,104.9940,105.0540,"100,000.0000",2,"105,024.0000",104.9940,NaN,NaN
2,21685WCJ4,2024-01-04,104.1050,0.0490,104.1050,104.1050,"50,000.0000",1,"52,052.5000",NaN,104.1050,NaN
3,21685WCJ4,2024-01-08,102.8700,0.0500,102.8700,102.8700,"1,000.0000",1,"1,028.7000",102.8700,NaN,NaN
4,21685WCJ4,2024-01-10,104.3552,0.0487,104.3260,104.3885,"30,000.0000",3,"31,306.5520",NaN,NaN,NaN


**Check against Dickerson.** WRDS hosts Alex Dickerson's cleaned daily corporate prices (same methodology, ends
March 2025). On overlapping bond-days our volume-weighted price should match his `prc_vw_par` closely.

In [12]:
dk = pull_dickerson(universe.loc[universe["asset_type"] == "corporate", "cusip"].tolist())
both = corp_daily.merge(dk, on=["cusip", "date"])
diff = (both["clean_price"] - both["dk_price_vw"]).abs()
print(f"{len(both):,} overlapping bond-days: median |diff| {diff.median():.4f} pts, 95th pct {diff.quantile(.95):.3f}, "
      f"within 0.25 pt: {(diff < 0.25).mean():.1%}")

data_cleaning.wrds_pull: dickerson_corporate_daily: 1404585 rows from cache


38,159 overlapping bond-days: median |diff| 0.0006 pts, 95th pct 0.078, within 0.25 pt: 99.0%


**Terms and accrued interest.** Coupon, maturity, frequency and day count come from Mergent FISD
(`fisd.fisd_mergedissue`), with the AGG file as fallback. Accrued interest (30/360 for almost all corporates) is
computed at the trade date, the same convention as CRSP's accrued for Treasuries. Dickerson accrues to T+2 settlement,
so his `dk_accrued_interest` sits about two days of coupon above ours.

In [13]:
terms = clean_terms(pull_fisd_terms(universe.loc[universe["asset_type"] == "corporate", "cusip"].tolist()), universe)
print(terms["terms_source"].value_counts().to_dict(), "| day count:", terms["day_count"].value_counts().to_dict(),
      "| coupon type:", terms["coupon_type"].value_counts(dropna=False).to_dict())
terms[["cusip", "maturity_date", "coupon_rate", "coupon_type", "coupon_frequency", "day_count", "dated_date",
       "first_interest_date", "min_denomination", "amount_outstanding", "is_callable", "terms_source"]].head()

data_cleaning.wrds_pull: fisd_corporate_terms: 8176 rows from cache


data_cleaning.corporates: corporate terms: 8838 bonds, 8176 from FISD


{'fisd': 8176, 'agg_file': 662} | day count: {'30/360': 8833, 'ACT/360': 5} | coupon type: {'F': 7368, 'V': 808, <NA>: 662}


,cusip,maturity_date,coupon_rate,coupon_type,coupon_frequency,day_count,dated_date,first_interest_date,min_denomination,amount_outstanding,is_callable,terms_source
0,20030NET4,2037-01-15,5.1700,<NA>,2,30/360,NaT,NaT,NaN,<NA>,<NA>,agg_file
1,161175CU6,2038-09-01,7.0900,<NA>,2,30/360,NaT,NaT,NaN,<NA>,<NA>,agg_file
2,03522AAJ9,2046-02-01,4.9000,F,2,30/360,2019-02-01,2019-08-01,"1,000.0000","9,518,964.0000",Y,fisd
3,87264ABF1,2030-04-15,3.8750,F,2,30/360,2021-04-15,2021-10-15,"2,000.0000","6,999,175.0000",Y,fisd
4,30303MAD4,2035-11-15,4.8750,F,2,30/360,2025-11-03,2026-05-15,"2,000.0000","6,500,000.0000",Y,fisd


## Step 4b: MBS pass-throughs

Same cleaning on the enhanced TRACE SPDS pool table (and the TBA table for AGG's 29 TBA positions). Two
differences: pools and TBAs settle on forward SIFMA dates, so the T+2 settlement filter does not apply, and TRACE
reports pool quantities in *original* face, which is scaled to current face with the pool factor on file.
MBS yields and spreads need a prepayment model and are left NaN.

In [14]:
raw_mbs = pd.read_parquet(sorted((C.RAW_DIR / "trace_mbs").glob("chunk_008_*.parquet"))[0])
mbs_trades, mbs_log = trace_clean.clean_trades(raw_mbs, product="mbs_pool")
mbs_daily = trace_clean.daily_prices(mbs_trades)
display(mbs_log)
mbs_daily[["cusip", "date", "clean_price", "price_low", "price_high", "par_volume", "trade_count", "pool_factor", "bid_ask_spread"]].head()

,step,trades_left
0,0 valid raw trades,246432
1,1 cancels/corrections/reversals,235950
2,2 inter-dealer duplicates,196391
3,3 non-standard trades,194834
4,4a median filter,193802
5,4b reversal filter,193776
6,5 pool factor available,193776


,cusip,date,clean_price,price_low,price_high,par_volume,trade_count,pool_factor,bid_ask_spread
0,31410F6B6,2024-04-11,103.7500,103.7500,103.7500,"132,762.8004",2,0.0090,NaN
1,31410F6B6,2024-05-06,104.4844,104.3438,104.6250,"270,117.6000",2,0.0090,NaN
2,31410F6B6,2024-05-07,106.0000,106.0000,106.0000,"133,143.0000",1,0.0089,NaN
3,31410F6B6,2024-06-25,105.0469,105.0469,105.0469,"25,623.0585",1,0.0088,NaN
4,31410F6B6,2024-08-22,103.2121,103.2121,103.2121,"6,821.3440",2,0.0085,NaN


## Step 5: yields and spreads

For every bond-day with bullet cash flows, `features.py` solves two numbers from the dirty price: the semiannual
yield to maturity, and the constant spread (bps) over that day's fitted zero curve that reprices the bond. Treasuries
get both too, which is a check on the curve (their spread should be about zero).

In [15]:
from data_cleaning.finmath import accrued_interest
tm = terms.set_index("cusip")
acc = np.full(len(corp_daily), np.nan)
for cusip, idx in corp_daily.groupby("cusip").indices.items():
    b = tm.loc[cusip]
    acc[idx] = accrued_interest(corp_daily["date"].values[idx], b["maturity_date"], b["coupon_rate"], int(b["coupon_frequency"]), b["day_count"])
corp_daily["accrued_interest"] = acc
corp_daily["dirty_price"] = corp_daily["clean_price"] + acc
feat = solve_yield_and_spread(corp_daily, terms, curve)
feat[["cusip", "date", "clean_price", "accrued_interest", "dirty_price", "yield", "ytm_solved", "spread_bps", "years_to_maturity"]].head()

data_cleaning.features: solved yields for 100.0% of rows, spreads for 88.6%


,cusip,date,clean_price,accrued_interest,dirty_price,yield,ytm_solved,spread_bps,years_to_maturity
0,21685WCJ4,2024-01-02,105.0344,0.5542,105.5885,0.0482,0.0482,66.5265,17.3908
1,21685WCJ4,2024-01-03,105.0240,0.5687,105.5927,0.0482,0.0482,67.3980,17.3881
2,21685WCJ4,2024-01-04,104.1050,0.5833,104.6883,0.0490,0.0490,65.7890,17.3854
3,21685WCJ4,2024-01-08,102.8700,0.6417,103.5117,0.0500,0.0500,73.7982,17.3744
4,21685WCJ4,2024-01-10,104.3552,0.6708,105.0260,0.0487,0.0488,60.3305,17.3689


In [16]:
# reported TRACE yield vs our solved yield (bps): small differences are settlement/convention effects
((feat["yield"] - feat["ytm_solved"]) * 1e4).describe().round(1)

count   77,819.0000
mean        -1.2000
std          9.6000
min       -400.5000
25%         -0.3000
50%         -0.1000
75%          0.1000
max          1.9000
dtype: float64

## Step 6: the finished tables

`run_pipeline.py` does the above for every chunk and writes parquet tables to `data/processed/`. The rest of this
notebook reads those.

In [17]:
P = C.PROCESSED_DIR
bond_universe = pd.read_parquet(P / "bond_universe.parquet")
prices = pd.read_parquet(P / "prices_daily.parquet")
yc = pd.read_parquet(P / "yield_curve_daily.parquet")
for name in ["bond_universe", "prices_daily", "treasury_daily", "corporate_daily", "mbs_daily", "yield_curve_daily"]:
    df = pd.read_parquet(P / f"{name}.parquet")
    print(f"{name:20s} {len(df):>10,} rows x {df.shape[1]:2d} cols   {', '.join(df.columns[:9])}, ...")

bond_universe            12,069 rows x 31 cols   cusip, name, sector, asset_type, agg_sector, is_tba, holdings_asof_date, agg_market_value, agg_par_value, ...


prices_daily          3,250,515 rows x 17 cols   cusip, date, asset_type, sector, source, clean_price, accrued_interest, dirty_price, yield, ...
treasury_daily          101,709 rows x 24 cols   cusip, date, security_type, is_zero_coupon, coupon_rate, dated_date, maturity_date, years_to_maturity, bid_price, ...


corporate_daily       3,083,297 rows x 36 cols   cusip, date, clean_price, yield, price_low, price_high, par_volume, trade_count, dollar_volume, ...
mbs_daily                65,509 rows x 21 cols   cusip, date, clean_price, yield, price_low, price_high, par_volume, trade_count, pool_factor, ...
yield_curve_daily           499 rows x 18 cols   date, zero_6m, zero_1y, zero_2y, zero_3y, zero_5y, zero_7y, zero_10y, zero_30y, ...


### `bond_universe`: one row per bond

AGG snapshot + terms + coverage. `agg_weight` is the market-value share of the fund; `model_weight` renormalises it
over the bonds that have at least one price, so it sums to 1 (this is the benchmark weight vector).

In [18]:
cov = bond_universe.groupby("asset_type").apply(lambda g: pd.Series({
    "bonds": len(g), "with_prices": int(g["has_prices"].sum()), "agg_weight_covered": g.loc[g["has_prices"], "agg_weight"].sum() / g["agg_weight"].sum(),
    "model_weight": g["model_weight"].sum(), "median_price_days": g.loc[g["has_prices"], "n_price_days"].median()}), include_groups=False)
cov.round(3)

,bonds,with_prices,agg_weight_covered,model_weight,median_price_days
asset_type,,,,,
corporate,"8,838.0000","8,033.0000",0.9060,0.2840,435.0000
mbs,"2,934.0000","1,794.0000",0.7580,0.2190,6.0000
treasury,297.0000,253.0000,0.8380,0.4970,499.0000


In [19]:
bond_universe[["cusip", "name", "sector", "agg_weight", "model_weight", "maturity_date", "coupon_rate", "coupon_frequency",
               "day_count", "terms_source", "min_denomination", "has_prices", "n_price_days", "first_price_date", "last_price_date"]].sample(6, random_state=0)

,cusip,name,sector,agg_weight,model_weight,maturity_date,coupon_rate,coupon_frequency,day_count,terms_source,min_denomination,has_prices,n_price_days,first_price_date,last_price_date
2709,247361ZT8,DELTA AIR LINES INC,industrial,0.0000,0.0000,2029-10-28,3.7500,2,30/360,fisd,"2,000.0000",True,543,2024-01-02,2026-03-17
9034,31307NP40,FGOLD 15YR,mbs_passthrough,0.0000,0.0000,2030-07-01,3.0000,12,30/360,agg_file,NaN,True,3,2024-02-09,2024-09-05
4155,438127AE2,HONDA MOTOR CO LTD,industrial,0.0000,0.0000,2030-07-08,4.6880,2,30/360,fisd,"2,000.0000",True,179,2025-06-30,2026-03-17
5229,589331AM9,MERCK & CO INC,industrial,0.0000,0.0000,2036-11-15,5.7500,2,30/360,fisd,"2,000.0000",True,356,2024-01-02,2026-03-16
3232,29379VAT0,ENTERPRISE PRODUCTS OPERATING LLC,industrial,0.0000,0.0000,2041-02-01,5.9500,2,30/360,fisd,"1,000.0000",True,457,2024-01-02,2026-03-17
5281,59156RBN7,METLIFE INC,financial,0.0000,0.0000,2045-03-01,4.0500,2,30/360,fisd,"2,000.0000",True,513,2024-01-02,2026-03-17


### `prices_daily`: the master daily table

One row per (cusip, date) on days with an observation: Treasuries every trading day, corporates and MBS on days they
traded. Nothing is forward-filled; month-end selection and stale-price handling belong to the model stage.

In [20]:
prices.groupby("asset_type").agg(rows=("date", "size"), bonds=("cusip", "nunique"), first=("date", "min"), last=("date", "max"),
                                 median_trades_per_day=("trade_count", "median"), yield_filled=("yield", lambda s: s.notna().mean()),
                                 spread_filled=("spread_bps", lambda s: s.notna().mean()), bid_ask_filled=("bid_ask_spread", lambda s: s.notna().mean())).round(2)

,rows,bonds,first,last,median_trades_per_day,yield_filled,spread_filled,bid_ask_filled
asset_type,,,,,,,,
corporate,3083297,8033,2024-01-02,2026-03-17,7.0000,0.9100,0.8900,0.6100
mbs,65509,1794,2024-01-02,2025-03-17,2.0000,0.0000,0.0000,0.1600
treasury,101709,253,2024-01-02,2025-12-31,NaN,1.0000,1.0000,1.0000


In [21]:
prices[prices["asset_type"] == "corporate"].sample(5, random_state=1)

,cusip,date,asset_type,sector,source,clean_price,accrued_interest,dirty_price,yield,ytm_solved,spread_bps,bid_ask_spread,bid_ask_spread_bps,par_volume,dollar_volume,trade_count,years_to_maturity
2011982,65339KCA6,2025-07-18,corporate,utility,trace_enhanced,61.0260,0.0250,61.0510,0.0594,0.0594,76.9354,NaN,NaN,"34,000.0000","20,748.8400",1.0000,26.4942
975166,254687EB8,2024-02-01,corporate,industrial,trace_enhanced,114.1045,0.8178,114.9223,0.0482,0.0483,78.6097,0.0000,0.0000,"1,611,000.0000","1,838,223.7500",7.0000,11.8686
1284413,35671DCF0,2024-10-04,corporate,industrial,trace_enhanced,97.2467,0.3896,97.6363,0.0483,0.0484,97.1265,0.4115,42.3130,"781,000.0000","759,497.0100",9.0000,5.4045
1218434,313747BA4,2024-12-04,corporate,financial,trace_enhanced,93.0338,0.0292,93.0630,0.0497,0.0497,85.7441,NaN,NaN,"230,000.0000","213,977.8000",2.0000,5.4894
1122061,29364GAK9,2024-06-13,corporate,utility,trace_enhanced,72.7430,1.8542,74.5972,0.0579,0.0579,125.2125,NaN,NaN,"70,000.0000","50,920.1000",1.0000,26.0041


In [22]:
prices.groupby("asset_type")[["clean_price", "accrued_interest", "ytm_solved", "spread_bps", "bid_ask_spread_bps", "dollar_volume"]].median().round(3)

,clean_price,accrued_interest,ytm_solved,spread_bps,bid_ask_spread_bps,dollar_volume
asset_type,,,,,,
corporate,97.0010,0.9970,0.0520,82.6570,13.0360,"955,838.0600"
mbs,90.5580,0.1280,NaN,NaN,26.2110,"141,723.0220"
treasury,94.9160,0.6780,0.0430,0.3200,4.8170,NaN


In [23]:
# how many bonds trade on a typical day, by sector: this drives how sparse the corporate/MBS panel is
daily_count = prices.groupby(["date", "asset_type"])["cusip"].size().unstack()
daily_count.describe().round(0)

asset_type,corporate,mbs,treasury
count,686.0000,328.0000,499.0000
mean,"4,495.0000",200.0000,204.0000
std,"2,344.0000",72.0000,26.0000
min,1.0000,1.0000,164.0000
25%,"4,580.0000",182.0000,182.0000
50%,"5,384.0000",211.0000,203.0000
75%,"6,046.0000",239.0000,226.0000
max,"7,406.0000",413.0000,253.0000


### `yield_curve_daily`: the key-rate zero yields

One row per CRSP trading day (also the trading calendar). Below: the curve at a few month-ends, and the
correlation of daily key-rate changes, which is what the level/slope/curvature factor model summarises.

Caveat: daily changes in the 30Y zero are noisier than the other tenors and only weakly correlated with them. Few bonds pin down the long end of the NSS fit, so some of that is fit noise rather than market movement; worth keeping in mind when estimating the key-rate covariance (e.g. shrinkage, or a shorter long-end anchor).

In [24]:
(yc.set_index("date")[KEY_COLS] * 100).iloc[::63].round(2)

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y
date,,,,,,,,
2024-01-02,5.0900,4.7400,4.2800,4.0400,3.8800,3.9000,3.9900,4.2000
2024-04-03,5.1700,4.9600,4.6300,4.4300,4.2900,4.2900,4.3700,4.6000
2024-07-03,5.2100,5.0100,4.6500,4.4300,4.2800,4.2800,4.3500,4.6700
2024-10-02,4.2700,3.9100,3.6000,3.5200,3.5400,3.6300,3.7800,4.4700
2025-01-06,4.2100,4.1900,4.2200,4.2800,4.3900,4.5000,4.6300,5.0600
2025-04-07,4.0900,3.8800,3.7400,3.7500,3.8700,4.0000,4.2100,5.0700
2025-07-09,4.2100,4.0400,3.8200,3.7800,3.9000,4.1000,4.3800,5.1800
2025-10-07,3.7800,3.6400,3.5400,3.5600,3.7100,3.9000,4.1700,5.1000


In [25]:
(yc.set_index("date")[KEY_COLS].diff().dropna().corr()).round(2)

,zero_6m,zero_1y,zero_2y,zero_3y,zero_5y,zero_7y,zero_10y,zero_30y
zero_6m,1.0000,0.8500,0.7000,0.6400,0.5900,0.5500,0.4900,0.1400
zero_1y,0.8500,1.0000,0.9300,0.8800,0.8200,0.7700,0.6900,0.1500
zero_2y,0.7000,0.9300,1.0000,0.9800,0.9100,0.8600,0.7900,0.1500
zero_3y,0.6400,0.8800,0.9800,1.0000,0.9600,0.9100,0.8300,0.2800
zero_5y,0.5900,0.8200,0.9100,0.9600,1.0000,0.9800,0.9000,0.4100
zero_7y,0.5500,0.7700,0.8600,0.9100,0.9800,1.0000,0.9700,0.3800
zero_10y,0.4900,0.6900,0.7900,0.8300,0.9000,0.9700,1.0000,0.2700
zero_30y,0.1400,0.1500,0.1500,0.2800,0.4100,0.3800,0.2700,1.0000


### Quick validity checks

The same checks run in `data_cleaning/validate.py`.

In [26]:
tsy = pd.read_parquet(P / "treasury_daily.parquet")
checks = {
    "one row per (cusip, date)": not prices.duplicated(["cusip", "date"]).any(),
    "model weights sum to 1": np.isclose(bond_universe["model_weight"].sum(), 1.0),
    "dirty >= clean": (prices["dirty_price"] >= prices["clean_price"] - 1e-9).all(),
    "curve complete every day": yc[KEY_COLS].notna().all().all(),
    "Treasury spread over own curve ~ 0 (median bps)": abs(tsy["spread_bps"].median()) < 2,
}
for k, ok in checks.items():
    print("PASS " if ok else "FAIL ", k)
print("\nTreasury spread over fitted curve (bps):", tsy["spread_bps"].describe()[["mean", "50%", "std"]].round(1).to_dict())

PASS  one row per (cusip, date)
PASS  model weights sum to 1
PASS  dirty >= clean
PASS  curve complete every day
PASS  Treasury spread over own curve ~ 0 (median bps)

Treasury spread over fitted curve (bps): {'mean': -0.1, '50%': 0.3, 'std': 5.2}
